In [31]:
import torch
from transformers import GPT2Config, GPT2Model, GPT2Tokenizer, GPT2LMHeadModel
from huggingface_hub import hf_hub_download
from dotenv import load_dotenv
from safetensors.torch import load_file
import os

In [2]:
config = GPT2Config()
gpt_model = GPT2Model(config)

In [3]:
print(gpt_model)

GPT2Model(
  (wte): Embedding(50257, 768)
  (wpe): Embedding(1024, 768)
  (drop): Dropout(p=0.1, inplace=False)
  (h): ModuleList(
    (0-11): 12 x GPT2Block(
      (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (attn): GPT2Attention(
        (c_attn): Conv1D(nf=2304, nx=768)
        (c_proj): Conv1D(nf=768, nx=768)
        (attn_dropout): Dropout(p=0.1, inplace=False)
        (resid_dropout): Dropout(p=0.1, inplace=False)
      )
      (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (mlp): GPT2MLP(
        (c_fc): Conv1D(nf=3072, nx=768)
        (c_proj): Conv1D(nf=768, nx=3072)
        (act): NewGELUActivation()
        (dropout): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
)


In [5]:
weights_path = hf_hub_download(
    repo_id="gpt2",
    filename="model.safetensors",
    token=os.getenv("HF_TOKEN")
)

In [7]:
print(weights_path)  # local path where it got cached, e.g. ~/.cache/huggingface/hub/...

/home/bakar_qureshi/.cache/huggingface/hub/models--gpt2/snapshots/607a30d783dfa663caf39e06633721c8d4cfcd7e/model.safetensors


In [9]:
state_dict = load_file(weights_path)

In [16]:
missing, unexpected = gpt_model.load_state_dict(state_dict, strict=False)

In [20]:
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

In [21]:
text = "what is car"

inputs = tokenizer(text, return_tensors="pt")

print(inputs)

{'input_ids': tensor([[10919,   318,  1097]]), 'attention_mask': tensor([[1, 1, 1]])}


In [23]:
gpt_model.eval()

with torch.no_grad():

    output = gpt_model(**inputs)

In [30]:
print(output.last_hidden_state.shape)

torch.Size([1, 3, 768])


In [32]:
gen_model = GPT2LMHeadModel.from_pretrained("gpt2")

[transformers] loading configuration file config.json from cache at /home/bakar_qureshi/.cache/huggingface/hub/models--gpt2/snapshots/607a30d783dfa663caf39e06633721c8d4cfcd7e/config.json
[transformers] Model config GPT2Config {
  "activation_function": "gelu_new",
  "add_cross_attention": false,
  "architectures": [
    "GPT2LMHeadModel"
  ],
  "attn_pdrop": 0.1,
  "bos_token_id": 50256,
  "embd_pdrop": 0.1,
  "eos_token_id": 50256,
  "initializer_range": 0.02,
  "layer_norm_epsilon": 1e-05,
  "model_type": "gpt2",
  "n_ctx": 1024,
  "n_embd": 768,
  "n_head": 12,
  "n_inner": null,
  "n_layer": 12,
  "n_positions": 1024,
  "pad_token_id": null,
  "reorder_and_upcast_attn": false,
  "resid_pdrop": 0.1,
  "scale_attn_by_inverse_layer_idx": false,
  "scale_attn_weights": true,
  "summary_activation": null,
  "summary_first_dropout": 0.1,
  "summary_proj_to_labels": true,
  "summary_type": "cls_index",
  "summary_use_proj": true,
  "task_specific_params": {
    "text-generation": {
      

In [34]:
gen_model.eval()

with torch.no_grad():

    output_ids = gen_model.generate(
        **inputs,
        max_new_tokens=20,
        do_sample=True,
        temperature=0.8,
        pad_token_id=tokenizer.eos_token_id
    )

In [ ]:
tokenizer.decode(output_ids[0], skip_special_tokens=True)

'what is car insurance?"'